### Requirements

In [ ]:
!pip install pyautogui

In [8]:
# Base Modules
import math
import time
import numpy as np

# OpenCV
import cv2

# MediaPipe
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PyAutoGUI
import pyautogui

## Parameters

In [28]:
# Camera Settings
camera_index = 700
resolution = [640, 480]
fps = 60

# Model Settings
model_path = '../models/hand_landmarker.task'
no_of_hands = 1
hand_confidence = 0.7
presence_confidence = 0.6
tracking_confidence = 0.6

# Gesture Settings
pinch_dist_max = 35
click_dist_max = 35
right_click_dist_max = 35

# Timing Settings
double_click_time = 0.65
drag_threshold_time = 0.35

# Screen & Cursor Settings
frame_margin = 90 
smoothing = 5

### Internal Parameters

In [29]:
# Get primary monitor resolution
screen_w, screen_h = pyautogui.size()
pyautogui.FAILSAFE = False  # Avoids crash if cursor bumps screen edges

# Tracking memory variables
prev_x, prev_y = screen_w // 2, screen_h // 2
is_clicked = False  # Debounce lock

# Tracking state variables
last_tap_time = 0
tap_count = 0
is_touching = False 
is_dragging = False
is_right_clicked = False

### MediaPipe

In [30]:
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=no_of_hands,
    min_hand_detection_confidence=hand_confidence,
    min_hand_presence_confidence=presence_confidence,
    min_tracking_confidence=tracking_confidence
)
detector = vision.HandLandmarker.create_from_options(options)

## Air Mouse Logic

In [31]:
def process_air_mouse(frame, result, w, h):
    global prev_x, prev_y, is_touching, last_tap_time, tap_count, is_dragging, is_right_clicked
    
    current_time = time.time()
    status_text = "Idle"
    
    # Active bounding box
    cv2.rectangle(frame, (frame_margin, frame_margin), 
                  (w - frame_margin, h - frame_margin), (200, 200, 200), 1)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Landmark 4 = Thumb | Landmark 8 = Index | Landmark 12 = Middle | Landmark 16 = Ring
        thumb_x, thumb_y = int(hand[4].x * w), int(hand[4].y * h)
        idx_x, idx_y = int(hand[8].x * w), int(hand[8].y * h)
        mid_x, mid_y = int(hand[12].x * w), int(hand[12].y * h)
        ring_x, ring_y = int(hand[16].x * w), int(hand[16].y * h)

        # Calculate gesture distances
        pinch_dist = math.hypot(thumb_x - idx_x, thumb_y - idx_y)         # Move check
        left_click_dist = math.hypot(idx_x - mid_x, idx_y - mid_y)        # Left click check
        right_click_dist = math.hypot(idx_x - ring_x, idx_y - ring_y)     # Right click check

        # Draw hand joints
        for lm in hand:
            cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, (0, 0, 255), cv2.FILLED)

        # ---------------- 1. CURSOR MOVEMENT ---------------- #
        if pinch_dist < pinch_dist_max:
            status_text = "Tracking Cursor"
            
            # Map index coordinates to screen resolution
            target_x = np.interp(idx_x, (frame_margin, w - frame_margin), (0, screen_w))
            target_y = np.interp(idx_y, (frame_margin, h - frame_margin), (0, screen_h))

            # Smoothing
            curr_x = prev_x + (target_x - prev_x) / smoothing
            curr_y = prev_y + (target_y - prev_y) / smoothing

            pyautogui.moveTo(curr_x, curr_y)
            prev_x, prev_y = curr_x, curr_y

            # Visuals
            cv2.line(frame, (thumb_x, thumb_y), (idx_x, idx_y), (0, 255, 0), 2)
            cv2.circle(frame, (idx_x, idx_y), 7, (0, 255, 0), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 7, (0, 255, 0), cv2.FILLED)

            # ---------------- 2. RIGHT CLICK (Ring touches Middle/Index) ---------------- #
            if right_click_dist < right_click_dist_max:
                if not is_right_clicked:
                    pyautogui.rightClick()
                    is_right_clicked = True
                status_text = "RIGHT CLICK!"
                cv2.line(frame, (idx_x, idx_y), (ring_x, ring_y), (0, 165, 255), 2) # Orange line
                cv2.circle(frame, (ring_x, ring_y), 9, (0, 165, 255), cv2.FILLED)

            # ---------------- 3. LEFT CLICK & DRAG ---------------- #
            elif left_click_dist < click_dist_max:
                is_right_clicked = False
                cv2.line(frame, (idx_x, idx_y), (mid_x, mid_y), (255, 255, 0), 2)
                cv2.circle(frame, (mid_x, mid_y), 9, (255, 255, 0), cv2.FILLED)

                if not is_touching:
                    is_touching = True
                    time_since_last_tap = current_time - last_tap_time

                    if time_since_last_tap <= double_click_time:
                        pyautogui.doubleClick()
                        tap_count = 0
                        last_tap_time = 0
                        status_text = "DOUBLE CLICK!"
                    else:
                        tap_count = 1
                        last_tap_time = current_time
                else:
                    if not is_dragging and (current_time - last_tap_time > drag_threshold_time):
                        pyautogui.mouseDown(button='left')
                        is_dragging = True
                    
                    if is_dragging:
                        status_text = "DRAGGING / HOLDING!"
            else:
                is_right_clicked = False
                if is_dragging:
                    pyautogui.mouseUp(button='left')
                    is_dragging = False
                    tap_count = 0
                is_touching = False

        else:
            # When tracking is released, reset all states
            if is_dragging:
                pyautogui.mouseUp(button='left')
                is_dragging = False
            is_touching = False
            is_right_clicked = False
            cv2.circle(frame, (idx_x, idx_y), 7, (0, 0, 255), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 7, (255, 0, 255), cv2.FILLED)

    # Resolve regular Single Left Click
    if tap_count == 1 and not is_dragging and (current_time - last_tap_time > double_click_time):
        pyautogui.click()
        tap_count = 0
        status_text = "SINGLE CLICK!"

    return frame, status_text

### Camera Configuration

In [32]:
cap = cv2.VideoCapture(camera_index)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, resolution[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, resolution[1])
cap.set(cv2.CAP_PROP_FPS, fps)

True

# Main Logic

In [ ]:
print("Air Mouse Active! Move inside the white box. Press 'q' to stop.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    # MediaPipe inference
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    timestamp_ms = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, timestamp_ms)

    # Process hand movements & click triggers
    frame, status_text = process_air_mouse(frame, result, w, h)

    # HUD Status
    is_active = "Tracking" in status_text or "CLICK" in status_text
    cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, 
                (0, 255, 0) if is_active else (0, 0, 255), 2)

    cv2.imshow("Air Mouse", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Air Mouse Active! Move inside the white box. Press 'q' to stop.
